# 1. Imports and Dummy Data / Model Setup
Imports the pipeline tools and creates a huge 2000x2000 fake tile to demonstrate.


In [4]:
%load_ext autoreload
%autoreload 2
%matplotlib inline
import sys
import os

# Insert project root at position 0 to prevent module shadowing from local check scripts
sys.path.insert(0, os.path.abspath('../'))


import torch
import torch.nn as nn
import numpy as np

sys.path.insert(0, os.path.abspath('../'))
from datasets.segmentation_ds import SegmentationDataset
from utils.spatial_ops import windowed_inference, windowed_evaluation

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# We use B=2 to test batching, and 2000x2000 to test sliding windows, keeping RAM usage low
B, H, W, C = 2, 2000, 2000, 3
NUM_CLASSES = 4

print(f"Generating {B} fake {H}x{W} images and masks...")
samples = np.random.rand(B, H, W, C).astype(np.float32)
masks = np.random.randint(0, NUM_CLASSES, (B, H, W)).astype(np.int32)


# dummy model
class DummyModel(nn.Module):
    def __init__(self, in_channels, num_classes):
        super().__init__()
        self.conv = nn.Conv2d(in_channels, num_classes, kernel_size=3, padding=1)
        
    def forward(self, x):
        return self.conv(x)

model = DummyModel(in_channels=C, num_classes=NUM_CLASSES).to(device)
model.eval()


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Using device: cuda
Generating 2 fake 2000x2000 images and masks...


DummyModel(
  (conv): Conv2d(3, 4, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
)

# 2. Data Loading with SegmentationDataset
Demonstrates stochastic grid sampling of large tiles over training epochs.


In [10]:
CROP_SIZE = 256
print("\nTesting SegmentationDataset...")

train_ds = SegmentationDataset(
    samples=samples, 
    masks=masks, 
    grid_sampling=True, 
    crop_size=CROP_SIZE,
    min_margin=128
)

# Simulate what happens across multiple epochs (shifts the grid for stochastic cropping)
print(f"Dataset length (number of crops): {len(train_ds)} | margin_x: {train_ds.margin_x}, margin_y: {train_ds.margin_y}")

for epoch in range(1, 4):
    train_ds.shift_grid(epoch=epoch)
    img_crop, mask_crop = train_ds[0]
    print(f"Epoch {epoch} | Stochastic Grid Offset (y, x): {train_ds.offsets.tolist()} | Crop Shape: {img_crop.shape}")



Testing SegmentationDataset...
Dataset length (number of crops): 98 | margin_x: 208, margin_y: 208
Epoch 1 | Stochastic Grid Offset (y, x): [[123, 179], [111, 168]] | Crop Shape: (256, 256, 3)
Epoch 2 | Stochastic Grid Offset (y, x): [[135, 3], [79, 15]] | Crop Shape: (256, 256, 3)
Epoch 3 | Stochastic Grid Offset (y, x): [[135, 189], [75, 0]] | Crop Shape: (256, 256, 3)


# 3. Artifact-Free Inference with windowed_inference
Runs the model safely over the massive 2000x2000 image by chunking it on GPU and merging it on CPU RAM.


In [6]:
print("\nTesting windowed_inference...")

# Convert the raw (B, H, W, C) images to a PyTorch tensor (B, C, H, W)
full_img_tensor = torch.tensor(samples).permute(0, 3, 1, 2).float()

with torch.no_grad():
    # merger_device="cpu" merges tiles in system RAM, preventing GPU VRAM OOM on massive outputs.
    pred_logits = windowed_inference(
        model=model, 
        image=full_img_tensor, 
        window_size=CROP_SIZE, 
        stride=CROP_SIZE // 2, 
        num_classes=NUM_CLASSES, 
        batch_size=16, 
        merger_device="cpu"
    )

print(f"Full stitched prediction shape: {pred_logits.shape}") 



Testing windowed_inference...
Full stitched prediction shape: torch.Size([2, 4, 2000, 2000])


# 5. Memory-Safe Evaluation
Calculates metrics smoothly by evaluating chunks iteratively.


In [7]:
print("\nTesting windowed_evaluation...")

pred_batch = pred_logits
mask_batch = torch.tensor(masks).long()

loss_fn = nn.CrossEntropyLoss()

class DummyMetric:
    """A dummy metric mimicking torchmetrics interface just to show it works"""
    def __init__(self):
        self.calls = 0
    def update(self, preds, target):
        self.calls += 1
        
dummy_metric = DummyMetric()

# Evaluate in 1024x1024 chunks to save memory
windowed_evaluation(
    preds=pred_batch, 
    masks=mask_batch, 
    tile_size=2000, 
    eval_batch_size=2,
    loss_fn=loss_fn, 
    torchmetrics=[dummy_metric]
)




Testing windowed_evaluation...


tensor(1.4323)